# 03 — Retrieval + Agent

Runs the end-to-end Apple Support prototype: intent classification, historical retrieval, grounded drafting, and escalation.

In [ ]:
import sys,os
from pathlib import Path
import pandas as pd
sys.path.insert(0,str(Path('..').resolve()))
from src.retrieval import Retriever
from src.model import IntentModel
from src.agent import escalate,reply
DATA=Path('../data')
if not DATA.exists(): DATA=Path('data')
h=pd.read_csv(DATA/'historical_support_pairs.csv').fillna('')
r=Retriever(h)
m=IntentModel()
m.fit(h.clean_message,h.conversation_id)


In [ ]:
def run_agent(message, conversation_id=None):
    intent,conf=m.predict(message)
    evidence=r.search(message,k=3,exclude_conversation_id=conversation_id)
    sim=evidence[0]['similarity'] if evidence else 0
    do_escalate,reason=escalate(message,sim,conf)
    text=('Escalate to a human support agent before sending a customer-facing response.' if do_escalate else reply(message,intent,evidence,os.getenv('OPENAI_API_KEY'),os.getenv('OPENAI_MODEL','gpt-4o-mini')))
    return {'message':message,'intent':intent,'confidence':conf,'top_similarity':sim,'escalate':do_escalate,'reason':reason,'reply':text}

examples=['My iPhone battery is draining after the latest update','My Apple ID will not let me sign in','Bluetooth keeps disconnecting']
for x in examples: print(run_agent(x)); print()
